In [18]:
import os
import pandas as pd


In [19]:
# Process a single .csv file to extract features and save it as '_processed.csv' in the same folder
def process_csv_file(file_path, folder):
    
    df = pd.read_csv(file_path)
    
    # extract file name and remove the extension
    file_name = os.path.splitext(os.path.basename(file_path))[0]
    df['ID'] = file_name

    # extract date and time features from the 'File' column using regex:
    # (\d{8}) captures 8 digits (date)
    # _ matches the underscore
    # (\d{6}) captures 6 digits (time)
    df[['Date_full', 'Time_full']] = df['File'].str.extract(r'(\d{8})_(\d{6})')
    
    df['Year'] = df['Date_full'].str[:4]
    df['Month'] = df['Date_full'].str[4:6]
    df['Day'] = df['Date_full'].str[6:]
    df['Time'] = df['Time_full'].str[:2]
    
    df.drop(columns=['Date_full', 'Time_full', 'File'], inplace=True)
    
    # define the output file path (saved in the same folder with a modified name)
    output_file_path = os.path.join(folder, f'{file_name}_processed.csv')

    df.to_csv(output_file_path, index=False)
    
    return output_file_path



In [20]:
# merge all processed files into one
def merge_processed_files(folder, merged_file_name='merged_processed.csv'):

    dataframes = []
    
    # iterate over the folder to find all files ending with '_processed.csv'
    for file_name in os.listdir(folder):
        if file_name.endswith('_processed.csv'):
            file_path = os.path.join(folder, file_name)
            df = pd.read_csv(file_path)
            dataframes.append(df)
    
    if dataframes:
        merged_df = pd.concat(dataframes, ignore_index=True)
        merged_file_path = os.path.join(folder, merged_file_name)
        merged_df.to_csv(merged_file_path, index=False)

        return merged_df

    return pd.DataFrame()



Confidence 0.5

In [22]:
folder = 'birdnet_field_data'

for file_name in os.listdir(folder):
    if file_name.endswith('.csv') and not file_name.endswith('_processed.csv'):
        file_path = os.path.join(folder, file_name)
        process_csv_file(file_path, folder)

merge_processed_files(folder)


,Start (s),End (s),Scientific name,Common name,Confidence,ID,Year,Month,Day,Time
0,0.0,3.0,Phylloscopus humei,Hume's Warbler,0.6411,psh10,2024,6,4,6
1,6.0,9.0,Phylloscopus humei,Hume's Warbler,0.8799,psh10,2024,6,4,6
2,12.0,15.0,Phylloscopus humei,Hume's Warbler,0.5355,psh10,2024,6,4,6
3,24.0,27.0,Phylloscopus humei,Hume's Warbler,0.5084,psh10,2024,6,4,6
4,27.0,30.0,Phylloscopus humei,Hume's Warbler,0.8052,psh10,2024,6,4,6
...,...,...,...,...,...,...,...,...,...,...
213109,87.0,90.0,Trochalopteron variegatum,Variegated Laughingthrush,0.7780,psm9,2024,7,26,7
213110,255.0,258.0,Thlypopsis ornata,Rufous-chested Tanager,0.5449,psm9,2024,7,26,7
213111,258.0,261.0,Streptoprocne zonaris,White-collared Swift,0.7196,psm9,2024,7,26,7
213112,315.0,318.0,Coccothraustes coccothraustes,Hawfinch,0.7357,psm9,2024,7,26,7


In [23]:
df05 = pd.read_csv('birdnet_field_data\merged_processed.csv')
df2 = pd.read_excel('map\loc_audio.xlsx')

merged05 = pd.merge(df05, df2, on='ID')



In [24]:
merged05.head()

,Start (s),End (s),Scientific name,Common name,Confidence,ID,Year,Month,Day,Time,Latitude,Longitude,Elevation
0,0.0,3.0,Phylloscopus humei,Hume's Warbler,0.6411,psh10,2024,6,4,6,33.13514,76.463139,3664
1,6.0,9.0,Phylloscopus humei,Hume's Warbler,0.8799,psh10,2024,6,4,6,33.13514,76.463139,3664
2,12.0,15.0,Phylloscopus humei,Hume's Warbler,0.5355,psh10,2024,6,4,6,33.13514,76.463139,3664
3,24.0,27.0,Phylloscopus humei,Hume's Warbler,0.5084,psh10,2024,6,4,6,33.13514,76.463139,3664
4,27.0,30.0,Phylloscopus humei,Hume's Warbler,0.8052,psh10,2024,6,4,6,33.13514,76.463139,3664


In [25]:

df_all_species = pd.read_excel('data/all_species.xlsx')

merged05['Scientific name'] = merged05['Scientific name'].str.lower()
merged05['Common name'] = merged05['Scientific name'].str.lower()

df_all_species['Scientific Name'] = df_all_species['Scientific Name'].str.lower()
df_all_species['Species Name'] = df_all_species['Scientific Name'].str.lower()

merged05['Habitant'] = merged05['Scientific name'].isin(df_all_species['Scientific Name']).astype(int)

merged05.head()

,Start (s),End (s),Scientific name,Common name,Confidence,ID,Year,Month,Day,Time,Latitude,Longitude,Elevation,Habitant
0,0.0,3.0,phylloscopus humei,phylloscopus humei,0.6411,psh10,2024,6,4,6,33.13514,76.463139,3664,1
1,6.0,9.0,phylloscopus humei,phylloscopus humei,0.8799,psh10,2024,6,4,6,33.13514,76.463139,3664,1
2,12.0,15.0,phylloscopus humei,phylloscopus humei,0.5355,psh10,2024,6,4,6,33.13514,76.463139,3664,1
3,24.0,27.0,phylloscopus humei,phylloscopus humei,0.5084,psh10,2024,6,4,6,33.13514,76.463139,3664,1
4,27.0,30.0,phylloscopus humei,phylloscopus humei,0.8052,psh10,2024,6,4,6,33.13514,76.463139,3664,1


In [26]:
merged05.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200608 entries, 0 to 200607
Data columns (total 14 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   Start (s)        200608 non-null  float64
 1   End (s)          200608 non-null  float64
 2   Scientific name  200608 non-null  object 
 3   Common name      200608 non-null  object 
 4   Confidence       200608 non-null  float64
 5   ID               200608 non-null  object 
 6   Year             200608 non-null  int64  
 7   Month            200608 non-null  int64  
 8   Day              200608 non-null  int64  
 9   Time             200608 non-null  int64  
 10  Latitude         200608 non-null  float64
 11  Longitude        200608 non-null  float64
 12  Elevation        200608 non-null  int64  
 13  Habitant         200608 non-null  int32  
dtypes: float64(5), int32(1), int64(5), object(3)
memory usage: 20.7+ MB


In [27]:
merged05.duplicated().sum()

0

In [28]:
merged05.ID.value_counts()

ID
psl2     18522
psl10    15055
psm4     13411
psl5     13178
psm6     13170
psl6     11951
psl4     11352
psl3      9752
psm8      9148
psl8      8463
psh3      8088
psm10     7584
psm7      6512
psm3      6512
psl9      6331
psl1      5738
psh4      5588
psh9      5440
psh5      5440
psm9      4958
psh7      4854
psl7      3582
psh2      2252
psh6      2155
psh10     1447
psh1       125
Name: count, dtype: int64

In [30]:
merged05.to_csv('data/data.csv')